In [1]:
import re
import pandas as pd
import html
import os
import jsonlines
import json
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

##### AI 허브 데이터 (한국어 성능이 개선된 초거대AI 언어모델 개발 및 데이터): https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=71748

In [2]:
train = "/home/work/Korean_Spoken/Benchmark/SFTlabel.json"
val = "/home/work/Korean_Spoken/Benchmark/SFTlabel_val.json"

In [4]:
q_list = []
c_list = []
ref_list = []


with open(train) as f:  # json 열기 
    test = json.load(f)
    f.close()

    for it in tqdm(test['data_info']):  
        if it['data_category']['main'] == "구어체":
            c_list.append(it['data_category']['middle'])
            q_list.append(it['question'])
            ref_list.append(it['answer']['contents'])
        else:
            pass

  0%|          | 0/10580 [00:00<?, ?it/s]

In [7]:
train_benchmark = pd.DataFrame({'Question': q_list, 
                                'Category': c_list,
                                'Reference': ref_list
                               })
train_benchmark.drop_duplicates(subset=['Question'], keep='first', inplace=True, ignore_index=True)
train_benchmark

,Question,Category,Reference
0,사상의학에서 유전자 분석은 어떻게 활용되고 있나요?,보건,사상의학에서 유전자 분석은 다양한 방법으로 활용되고 있습니다. 일반적으로 질병 예방...
1,도자기를 고르게 돌리기 위해 사용하는 도구의 이름은?,예체능,도자기를 고르게 돌리기 위해 사용하는 도구는 '도마'입니다. 도마는 도자기를 고르게...
2,동물 질병 예방을 위한 접종을 어떻게 진행하는가?,보건,동물 질병 예방을 위한 접종은 동물의 종류와 질병에 따라 다소 차이가 있을 수 있지...
3,높은 곳에서 떨어질 때 왜 속도가 점점 더 빨라지나요?,자연,높은 곳에서 물체가 떨어질 때 속도가 점점 빨라지는 이유는 중력입니다. \n\n지구...
4,강생이랑 궤 중에서 아파트에서 키우기 좋은 종류는 무엇입니까?,자연,"강아지는 아파트에서 키우기 좋은 종류로 포메라니안, 믹스견, 시츄, 비글, 몰티즈,..."
...,...,...,...
4664,큐레이터가 뭐 하는 사람이에요? 어떤 일을 하나요?,예체능,"큐레이터는 특정 주제나 분야에 대해 연구하고, 컬렉션을 구성하고, 전시를 기획하는 ..."
4665,오븐 사용 시 음식이 빨리 익히려면 어떤 기능을 사용하면 좋을까요?,기타,오븐을 사용하여 음식을 빨리 익히는 방법에는 몇 가지가 있습니다. 다음은 일반적인 ...
4666,"눈, 귀, 입, 손, 발과 같은 몸의 부분을 표현할 때 어떤 조사를 사용해야 하는가?",인문,"한국어에서 몸의 부분을 표현할 때에는 ""의""라는 조사를 사용합니다. ""의"" 조사는 ..."
4667,저해상도 TV 대비 고해상도 TV의 장단점을 설명하시오.,공학,고해상도 TV의 장점은 다음과 같습니다:\n\n1. 더 선명하고 세밀한 화질: 고해...


In [8]:
train_benchmark.to_csv("/home/work/Korean_Spoken/Benchmark/train_benchmark.csv")

### Val

In [9]:
q_list = []
c_list = []
ref_list = []


with open(val) as f:  # json 열기 
    test = json.load(f)
    f.close()

    for it in tqdm(test['data_info']):  
        if it['data_category']['main'] == "구어체":
            c_list.append(it['data_category']['middle'])
            q_list.append(it['question'])
            ref_list.append(it['answer']['contents'])
        else:
            pass

  0%|          | 0/1322 [00:00<?, ?it/s]

In [10]:
val_benchmark = pd.DataFrame({'Question': q_list, 
                                'Category': c_list,
                                'Reference': ref_list
                               })
val_benchmark.drop_duplicates(subset=['Question'], keep='first', inplace=True, ignore_index=True)
val_benchmark

,Question,Category,Reference
0,구름을 보고 느끼는 감정에 대한 주제로 시를 써줘.,기타,한여름 밤 구름\n야자수 그늘을 찾아 \n또 하나의 이야기를 품고 있는 \n\n구름...
1,유리공예 작업할 때 필요한 도구와 재료가 뭐야?,예체능,유리공예 작업을 위해 필요한 도구 및 재료는 다음과 같습니다:\n\n도구\n1. 유...
2,산맥에서 어떤 종류의 동식물이 발견될까요?,자연,"산맥 지역에서는 다양한 종류의 동식물이 발견됩니다. 이는 지역의 기후, 고도, 지형..."
3,제주도 사투리가 다른 지방 사투리와 함께 사용되는 경우가 있나요?,기타,"네, 제주도 사투리는 다른 지방 사투리와 함께 사용되는 경우도 있습니다. 특히 한국..."
4,IoT (사물인터넷)은 어떻게 연결되는가?,공학,IoT는 다양한 디바이스나 센서들이 인터넷을 통해 서로 연결되는 것을 의미합니다. ...
...,...,...,...
556,가공업에서의 생산량 감소 시 조치하는 방법은 무엇인가요?,산업,가공업에서 생산량이 감소하는 경우에는 다양한 조치를 취할 수 있습니다. 주요한 방법...
557,시집 이름을 선택하면서 어떤 키워드를 고려하고 있어?,명칭,시집 이름을 선택할 때 고려할 수 있는 몇 가지 키워드는 다음과 같습니다:\n\n1...
558,붉은 귀 거북이는 얼마나 오래 사나요?,자연,붉은 귀 거북은 약 30년에서 50년 정도 수명이 있다고 알려져 있습니다. 하지만 ...
559,한국 미술에서 사용되는 전통적인 재료와 기술은 무엇인가?,예체능,한국 미술에서 사용되는 전통적인 재료와 기술은 다양합니다. 일부 예시로는 다음과 같...


In [17]:
val_benchmark['Category'].unique()

array(['기타', '예체능', '자연', '공학', '산업', '사회', '명칭', '인문', '보건', '종교'],
      dtype=object)

In [23]:
val_benchmark[val_benchmark['Category'] == '기타']

,Question,Category,Reference
0,구름을 보고 느끼는 감정에 대한 주제로 시를 써줘.,기타,한여름 밤 구름\n야자수 그늘을 찾아 \n또 하나의 이야기를 품고 있는 \n\n구름...
3,제주도 사투리가 다른 지방 사투리와 함께 사용되는 경우가 있나요?,기타,"네, 제주도 사투리는 다른 지방 사투리와 함께 사용되는 경우도 있습니다. 특히 한국..."
12,야민 정음을 통해 어떤 종류의 문화 콘텐츠가 생겨나고 있는지 설명하시오.,기타,"야민 정음은 한국어에서 일부 알파벳과 숫자, 문자 등을 형태소 단위로 나누어 다른 ..."
18,경조사에서 선물 포장과 카드 작성을 어떻게 꾸미나요?,기타,"선물 포장과 카드 작성은 기본적으로 개인의 취향과 상황에 따라 다를 수 있지만, 일..."
20,나는 소파 색상과 디자인이 방의 분위기에 어떤 영향을 미치는지 궁금하다.,기타,소파의 색깔과 디자인은 방 분위기에 큰 영향을 미칩니다. 이들은 방의 전반적인 분위...
...,...,...,...
523,조개 껍질은 일반 쓰레기로 버려야 하나요?,기타,조개 껍데기는 일반 쓰레기로 버려야 합니다. 조개 껍데기는 자연 소재이기 때문에 재...
525,재미있는 퀴즈 내 줘.,기타,"네, 재미있는 퀴즈를 내보겠습니다. '왕이 넘어지면?'"
531,가구를 받을 때 무엇을 봐야 하나요?,기타,가구를 배송받을 때 주의해야 할 점은 다음과 같습니다:\n\n1. 배송일자와 시간 ...
535,태풍의 눈은 무엇이고 특징은 무엇입니까?,기타,태풍의 눈은 태풍의 중심 부근에 형성되는 원형의 영역을 말합니다. 태풍의 눈은 대기...


In [11]:
val_benchmark.to_csv("/home/work/Korean_Spoken/Benchmark/val_benchmark.csv")